In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
!unzip "/content/drive/MyDrive/reid_dataset.zip"

Streaming output truncated to the last 5000 lines.
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000567.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000386.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000575.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000436.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000396.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000558.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000749.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000253.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000198.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000027.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000447.jpg  
  inflating: content/reid_dataset/SNMOT-153/player_0007/frame_000118.jpg  
  inflating: content/reid_dataset/SNMOT-153/playe

In [ ]:
!pip install timm torch torchvision


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import timm
from torch.optim import lr_scheduler
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm


In [ ]:
# =======================
# إعدادات عامة
# =======================
data_dir = '/content/content/reid_dataset'
batch_size = 32
epochs = 5
lr = 3e-4
img_size = 224
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# تقسيم ثابت 80% / 20% بدون عشوائية
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset = Subset(dataset, list(range(0, train_size)))
val_dataset = Subset(dataset, list(range(train_size, len(dataset))))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

num_classes = len(dataset.classes)
print(f"Number of players (IDs): {num_classes}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


Number of players (IDs): 57
Number of training images: 586400
Number of validation images: 146601
classes : ['SNMOT-060', 'SNMOT-061', 'SNMOT-062', 'SNMOT-063', 'SNMOT-064', 'SNMOT-065', 'SNMOT-066', 'SNMOT-067', 'SNMOT-068', 'SNMOT-069', 'SNMOT-070', 'SNMOT-071', 'SNMOT-072', 'SNMOT-073', 'SNMOT-074', 'SNMOT-075', 'SNMOT-076', 'SNMOT-077', 'SNMOT-097', 'SNMOT-098', 'SNMOT-099', 'SNMOT-100', 'SNMOT-101', 'SNMOT-102', 'SNMOT-103', 'SNMOT-104', 'SNMOT-105', 'SNMOT-106', 'SNMOT-107', 'SNMOT-108', 'SNMOT-109', 'SNMOT-110', 'SNMOT-111', 'SNMOT-112', 'SNMOT-113', 'SNMOT-114', 'SNMOT-115', 'SNMOT-151', 'SNMOT-152', 'SNMOT-153', 'SNMOT-154', 'SNMOT-155', 'SNMOT-156', 'SNMOT-157', 'SNMOT-158', 'SNMOT-159', 'SNMOT-160', 'SNMOT-161', 'SNMOT-162', 'SNMOT-163', 'SNMOT-164', 'SNMOT-165', 'SNMOT-166', 'SNMOT-167', 'SNMOT-168', 'SNMOT-169', 'SNMOT-170']


In [ ]:
# =======================
# Model Definition
# =======================
class ReIDModel(nn.Module):
    def __init__(self, backbone_name='vit_base_patch16_224', embed_dim=512, num_classes=100):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        in_features = self.backbone.num_features
        self.embedding = nn.Linear(in_features, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        emb = self.embedding(feats)
        emb = self.bn(emb)
        logits = self.classifier(emb)
        return emb, logits

model = ReIDModel(num_classes=num_classes).to(device)


In [ ]:
# =======================
# Losses and Optimizer
# =======================
criterion_id = nn.CrossEntropyLoss()
criterion_triplet = nn.TripletMarginLoss(margin=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
scaler = GradScaler()

/tmp/ipython-input-4095456417.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# =======================
# Helper function: sample triplets
# =======================
import random

def get_triplets(batch_emb, batch_labels):
    anchors, positives, negatives = [], [], []
    for i in range(len(batch_labels)):
        anchor_label = batch_labels[i]
        pos_idx = (batch_labels == anchor_label).nonzero(as_tuple=True)[0]
        neg_idx = (batch_labels != anchor_label).nonzero(as_tuple=True)[0]
        if len(pos_idx) > 1 and len(neg_idx) > 0:
            pos = random.choice(pos_idx[pos_idx != i])
            neg = random.choice(neg_idx)
            anchors.append(batch_emb[i])
            positives.append(batch_emb[pos])
            negatives.append(batch_emb[neg])
    if len(anchors) > 0:
        return torch.stack(anchors), torch.stack(positives), torch.stack(negatives)
    else:
        return None, None, None


In [ ]:
best_val_loss = float('inf')

for epoch in range(epochs):
    # =======================
    # TRAIN
    # =======================
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):  # ✅ الصيغة الجديدة
            emb, logits = model(imgs)
            loss_id = criterion_id(logits, labels)
            anc, pos, neg = get_triplets(emb, labels)
            if anc is not None:
                loss_triplet = criterion_triplet(anc, pos, neg)
                loss = loss_id + loss_triplet
            else:
                loss = loss_id

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # =======================
    # VALIDATION
    # =======================
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            emb, logits = model(imgs)
            loss_id = criterion_id(logits, labels)
            val_loss += loss_id.item()

            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} "
          f"| Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    # حفظ أفضل موديل
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/reid_model_best.pth")
        print("✅ New best model saved!")

torch.save(model.state_dict(), "/content/reid_model_final.pth")
print("✅ Training finished.")


In [ ]:
import timm
print(timm.list_models('*vit*'))


['convit_base', 'convit_small', 'convit_tiny', 'crossvit_9_240', 'crossvit_9_dagger_240', 'crossvit_15_240', 'crossvit_15_dagger_240', 'crossvit_15_dagger_408', 'crossvit_18_240', 'crossvit_18_dagger_240', 'crossvit_18_dagger_408', 'crossvit_base_240', 'crossvit_small_240', 'crossvit_tiny_240', 'davit_base', 'davit_base_fl', 'davit_giant', 'davit_huge', 'davit_huge_fl', 'davit_large', 'davit_small', 'davit_tiny', 'efficientvit_b0', 'efficientvit_b1', 'efficientvit_b2', 'efficientvit_b3', 'efficientvit_l1', 'efficientvit_l2', 'efficientvit_l3', 'efficientvit_m0', 'efficientvit_m1', 'efficientvit_m2', 'efficientvit_m3', 'efficientvit_m4', 'efficientvit_m5', 'fastvit_ma36', 'fastvit_mci0', 'fastvit_mci1', 'fastvit_mci2', 'fastvit_mci3', 'fastvit_mci4', 'fastvit_s12', 'fastvit_sa12', 'fastvit_sa24', 'fastvit_sa36', 'fastvit_t8', 'fastvit_t12', 'flexivit_base', 'flexivit_large', 'flexivit_small', 'gcvit_base', 'gcvit_small', 'gcvit_tiny', 'gcvit_xtiny', 'gcvit_xxtiny', 'levit_128', 'levit_1